In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["TeX Gyre Pagella", "Book Antiqua", "Palatino Linotype", "DejaVu Serif"],
})

### Load one (cluster, sim, projection) sample
Each sample has a Stage-1 `intermediate/sample_NNNNNN.npz` (truth $\kappa_\infty$, binned observed shear $\hat\gamma_{1,2}$, source-galaxy count map) and a Stage-2b `posterior/posterior_NNNNNN.npz` (DLPosterior mean and std of the reconstructed $\kappa_E$).

In [ ]:
DATA_ROOT = "/projects/mccleary_group/habjan.e/TNG/Data/jaxlense_dataset"
idx = 1  # change to view a different sample

interm = np.load(os.path.join(DATA_ROOT, f"intermediate/sample_{idx:06d}.npz"))
post = np.load(os.path.join(DATA_ROOT, f"posterior/posterior_{idx:06d}.npz"))

kappa_inf = interm["kappa_inf"]
gamma1 = interm["gamma1_obs"]
gamma2 = interm["gamma2_obs"]
kappa_mean = post["kappa_E_mean"]
kappa_std = post["kappa_E_std"]

print(f"sample {idx}: cluster={int(interm['cluster_idx'])}, sim={str(interm['sim'])}, "
      f"M200~{float(interm['halo_mass_msun']):.2e} Msun, "
      f"n_member={int(interm['n_member'])}, n_source={int(interm['n_source'])}, "
      f"<beta>={float(post['mean_beta']):.3f}, "
      f"n_posterior_samples={int(post['n_posterior_samples'])}")

### True $\kappa_\infty$ ($z_s=\infty$) and the DLPosterior $\kappa_E$ posterior mean / std

In [ ]:
panels = [
    (kappa_inf,  "cubehelix", r"$\kappa_\infty$ (truth)",  (np.quantile(kappa_inf, 0.5),  np.quantile(kappa_inf, 0.98))),
    #(gamma1,     "RdBu_r",    r"$\hat\gamma_1$",            (-np.quantile(np.abs(gamma1), 0.98), np.quantile(np.abs(gamma1), 0.98))),
    #(gamma2,     "RdBu_r",    r"$\hat\gamma_2$",            (-np.quantile(np.abs(gamma2), 0.98), np.quantile(np.abs(gamma2), 0.98))),
    (kappa_mean, "cubehelix", r"$\kappa_E$ posterior mean", (np.quantile(kappa_mean, 0.5), np.quantile(kappa_mean, 0.98))),
    (kappa_std,  "cubehelix", r"$\kappa_E$ posterior std",  (0.0, np.quantile(kappa_std, 0.98))),
]

fig, axes = plt.subplots(nrows=1, ncols=len(panels), figsize=(4.0 * len(panels), 4.5),
                         constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.2, h_pad=0.2, wspace=0.1, hspace=0.1)

for ax, (img, cmap, label, (vmin, vmax)) in zip(axes, panels):
    im = ax.imshow(img, vmin=vmin, vmax=vmax, cmap=cmap, origin="upper")
    ax.set_xlabel("X-coordinate", fontsize=14, fontweight="semibold")
    ax.set_ylabel("Y-coordinate", fontsize=14, fontweight="semibold")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    cbar.set_label(label, fontsize=14, fontweight="semibold")

#fig.savefig("/home/habjan.e/TNG/cluster_deprojection/figures/jaxlensing_data.png", bbox_inches="tight")
plt.show()

### binned observed shear $\hat\gamma_1,\hat\gamma_2$

In [ ]:
panels = [
    (gamma1,     "RdBu_r",    r"$\hat\gamma_1$",            (-np.quantile(np.abs(gamma1), 0.98), np.quantile(np.abs(gamma1), 0.98))),
    (gamma2,     "RdBu_r",    r"$\hat\gamma_2$",            (-np.quantile(np.abs(gamma2), 0.98), np.quantile(np.abs(gamma2), 0.98))),
]

fig, axes = plt.subplots(nrows=1, ncols=len(panels), figsize=(4.0 * len(panels), 4.5),
                         constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.2, h_pad=0.2, wspace=0.1, hspace=0.1)

for ax, (img, cmap, label, (vmin, vmax)) in zip(axes, panels):
    im = ax.imshow(img, vmin=vmin, vmax=vmax, cmap=cmap, origin="upper")
    ax.set_xlabel("X-coordinate", fontsize=14, fontweight="semibold")
    ax.set_ylabel("Y-coordinate", fontsize=14, fontweight="semibold")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    cbar.set_label(label, fontsize=14, fontweight="semibold")

#fig.savefig("/home/habjan.e/TNG/cluster_deprojection/figures/jaxlensing_data.png", bbox_inches="tight")
plt.show()

### Kaiser & Squires 1993 inversion

In [ ]:
import sys
sys.path.append('/home/habjan.e/TNG/Codes/jax-lensing')

from jax_lensing.inversion import ks93


kE, kB = ks93(interm["gamma1_obs"], interm["gamma2_obs"])

### Plot true convergence $\kappa_E$ and $\kappa_B$

In [ ]:
panels = [
    (kappa_inf,  "cubehelix", r"$\kappa_\infty$ (truth)",  (np.quantile(kappa_inf, 0.5),  np.quantile(kappa_inf, 0.98))),
    #(gamma1,     "RdBu_r",    r"$\hat\gamma_1$",            (-np.quantile(np.abs(gamma1), 0.98), np.quantile(np.abs(gamma1), 0.98))),
    #(gamma2,     "RdBu_r",    r"$\hat\gamma_2$",            (-np.quantile(np.abs(gamma2), 0.98), np.quantile(np.abs(gamma2), 0.98))),
    (kE, "cubehelix", r"$\kappa_E$ KS93", (np.quantile(kE, 0.5), np.quantile(kE, 0.98))),
    (kB,  "cubehelix", r"$\kappa_B$ KS93",  (0.0, np.quantile(kB, 0.98))),
]

fig, axes = plt.subplots(nrows=1, ncols=len(panels), figsize=(4.0 * len(panels), 4.5),
                         constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.2, h_pad=0.2, wspace=0.1, hspace=0.1)

for ax, (img, cmap, label, (vmin, vmax)) in zip(axes, panels):
    im = ax.imshow(img, vmin=vmin, vmax=vmax, cmap=cmap, origin="upper")
    ax.set_xlabel("X-coordinate", fontsize=14, fontweight="semibold")
    ax.set_ylabel("Y-coordinate", fontsize=14, fontweight="semibold")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    cbar.set_label(label, fontsize=14, fontweight="semibold")

fig.savefig("/home/habjan.e/TNG/cluster_deprojection/figures/ks93_data.png", bbox_inches="tight")
plt.show()